In [5]:

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

# ==========================================
# 1. Definir os caminhos dos arquivos
# ==========================================

arquivo_microdados = Path(
    "01_PENSE2024_MICRODADOS.csv"
)

arquivo_recomendadas = Path(
    "01_Colunas_recomendadas_pense_2024_final.xlsx"
)

arquivo_saida = Path(
    "02_PENSE2024_MICRODADOS_FILTRADO.parquet"
)

# ==========================================
# 2. Ler a planilha de colunas recomendadas
# ==========================================

df_recomendadas = pd.read_excel(
    arquivo_recomendadas
)

codigos_recomendados = set(
    df_recomendadas["Código da coluna"]
    .dropna()
    .astype(str)
    .str.strip()
)

# ==========================================
# 3. Identificar as colunas do CSV
# ==========================================

colunas_microdados = pd.read_csv(
    arquivo_microdados,
    sep=";",
    encoding="utf-8-sig",
    nrows=0
).columns.tolist()

colunas_mantidas = [
    coluna
    for coluna in colunas_microdados
    if coluna.strip() in codigos_recomendados
]

codigos_ausentes = (
    codigos_recomendados - set(colunas_microdados)
)

# ==========================================
# 4. Processar e gravar em Parquet
# ==========================================

escritor = None
total_linhas = 0

try:

    for bloco in pd.read_csv(
        arquivo_microdados,
        sep=";",
        encoding="utf-8-sig",
        usecols=colunas_mantidas,
        chunksize=10000,
        low_memory=False
    ):

        # Converter o bloco para tabela Arrow
        tabela = pa.Table.from_pandas(
            bloco,
            preserve_index=False
        )

        # Criar o escritor no primeiro bloco
        if escritor is None:

            escritor = pq.ParquetWriter(
                arquivo_saida,
                tabela.schema,
                compression="snappy"
            )

        # Gravar o bloco no arquivo Parquet
        escritor.write_table(tabela)

        total_linhas += len(bloco)

        print(
            f"Linhas processadas: {total_linhas:,}"
        )

finally:

    if escritor is not None:
        escritor.close()

# ==========================================
# 5. Relatório final
# ==========================================

print("\n" + "=" * 50)
print("PROCESSAMENTO CONCLUÍDO")
print("=" * 50)

print(
    f"Total de linhas: {total_linhas:,}"
)

print(
    f"Colunas originais: {len(colunas_microdados)}"
)

print(
    f"Colunas mantidas: {len(colunas_mantidas)}"
)

print(
    f"Colunas removidas: "
    f"{len(colunas_microdados) - len(colunas_mantidas)}"
)

print(
    f"Códigos recomendados não encontrados: "
    f"{len(codigos_ausentes)}"
)

print(f"Arquivo gerado: {arquivo_saida}")

print(
    f"Tamanho do arquivo: "
    f"{arquivo_saida.stat().st_size / (1024**2):.2f} MB"
)

print("=" * 50)


Linhas processadas: 10,000
Linhas processadas: 20,000
Linhas processadas: 30,000
Linhas processadas: 40,000
Linhas processadas: 50,000
Linhas processadas: 60,000
Linhas processadas: 70,000
Linhas processadas: 80,000
Linhas processadas: 90,000
Linhas processadas: 100,000
Linhas processadas: 110,000
Linhas processadas: 120,000
Linhas processadas: 130,000
Linhas processadas: 140,000
Linhas processadas: 150,000
Linhas processadas: 153,870

PROCESSAMENTO CONCLUÍDO
Total de linhas: 153,870
Colunas originais: 430
Colunas mantidas: 41
Colunas removidas: 389
Códigos recomendados não encontrados: 0
Arquivo gerado: 02_PENSE2024_MICRODADOS_FILTRADO.parquet
Tamanho do arquivo: 1.77 MB


In [6]:
import pandas as pd

# Ler o arquivo Parquet
df = pd.read_parquet("02_PENSE2024_MICRODADOS_FILTRADO.parquet")

# Imprimir os códigos em ordem alfabética
for coluna in sorted(df.columns):
    print(coluna)

# Total de colunas
print("\nTotal:", len(df.columns))

B01001A
B01002
B01006
B01007
B01015B
B01016
B03006B
B03010C
B05002A
B07002
B07004
B07005
B07006
B07007B
B07009
B07011A
B07012A
B07013
B07016
B09001
B09002
B09003A
B09010A1
B09016A1
B09016A2
B12003
B12004
B12005
B12006
B12007
B12008
B12009
CAPITAL
E01P117
E01P11801
E01P11802
E01P8505
E01P8510
IDADE_AGREG
REGIAO
UF

Total: 41
